# R26 Analysis Backend — Run in Google Colab

Trains the **next-month sales & cost** forecaster (`HistGradientBoosting`, same logic as `Train_In_Colab.ipynb`) and builds `models/business_forecaster.joblib`.

**Before you run:** **Runtime → Run all**, or run cells top to bottom.

Edit **one** of: Git clone URL (next cell) **or** use the ZIP upload section later.

## 1) Get `Analysis_Backend` on Colab

**Option A — GitHub:** set `REPO_URL` to your repo (must contain `Analysis_Backend/`).

In [ ]:
# ---- EDIT if needed: your GitHub repo (no trailing slash) ----
REPO_URL = "https://github.com/YOUR_USERNAME/R26-IT-026.git"

import os
if "YOUR_USERNAME" in REPO_URL:
    raise SystemExit("Edit REPO_URL above to your real GitHub path, then re-run this cell.")

import os, subprocess
if os.path.isdir("R26-IT-026"):
    print("Folder exists; set REMOVE_EXISTING=True in this cell to re-clone")
REMOVE_EXISTING = False
if REMOVE_EXISTING and os.path.isdir("R26-IT-026"):
    !rm -rf R26-IT-026
if not os.path.isdir("R26-IT-026"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "R26-IT-026"], check=True)
%cd R26-IT-026/Analysis_Backend
!ls -la

**Option B — ZIP (if you skip Git):** zip the **`Analysis_Backend`** folder on your PC, then run the next cell, upload the ZIP, then continue from “Install dependencies”.

In [ ]:
# Uncomment to upload Analysis_Backend.zip, then %cd into it manually if path differs
# from google.colab import files
# uploaded = files.upload()
# !unzip -o Analysis_Backend.zip -d /content/
# %cd /content/Analysis_Backend
# !ls

print("If you used ZIP: run the unzip + %cd lines above, then skip the git clone cell.")

## 2) Data file `data/monthly_performance.csv`

Required columns: **`month`**, **`sales`**, **`cost`** (e.g. `2000-01`).  
If the repo already has `data/monthly_performance.csv`, the next cell only checks it.  
If missing, run the **upload** cell after that.

In [ ]:
from pathlib import Path
csv_path = Path("data/monthly_performance.csv")
if csv_path.is_file():
    print("OK:", csv_path.resolve())
    !head -5 {csv_path}
else:
    print("MISSING:", csv_path, "— run the next cell to upload CSV.")

In [ ]:
# Run only if data/monthly_performance.csv is missing
from google.colab import files
!mkdir -p data
uploaded = files.upload()
import shutil
for name in uploaded.keys():
    if name.endswith(".csv"):
        shutil.move(name, "data/monthly_performance.csv")
        break
!head -5 data/monthly_performance.csv

## 3) Install Python dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4) Train and save `models/business_forecaster.joblib`

In [ ]:
!python train_model.py

## 5) Download the model to your computer

Save the file into your project: **`Analysis_Backend/models/business_forecaster.joblib`**  
Then on your PC: `uvicorn app.main:app --host 0.0.0.0 --port 8000`

In [ ]:
from google.colab import files
files.download("models/business_forecaster.joblib")

## 6) (Optional) One prediction in-notebook

In [ ]:
import json
from pathlib import Path
from app.forecaster import predict_next_month_from_csv

out = predict_next_month_from_csv(Path("data/monthly_performance.csv"), Path("models/business_forecaster.joblib"))
print(json.dumps(out, indent=2))

## 7) (Optional) Public FastAPI URL with ngrok

Stops when the Colab runtime disconnects. Optional: Colab **Secrets** → add `NGROK_AUTHTOKEN` from https://dashboard.ngrok.com

In [ ]:
!pip install -q pyngrok

import threading
import uvicorn

try:
    from google.colab import userdata
    t = userdata.get("NGROK_AUTHTOKEN")
    if t:
        !ngrok config add-authtoken {t}
except Exception:
    pass

from pyngrok import ngrok
tunnel = ngrok.connect(8000)
print("API docs:", str(tunnel.public_url).rstrip("/") + "/docs")

def run():
    uvicorn.run("app.main:app", host="0.0.0.0", port=8000)

threading.Thread(target=run, daemon=True).start()